In [ ]:
import pyscrew
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.utils import shuffle 
from sklearn.preprocessing import LabelEncoder
from tabpfn import TabPFNClassifier
import time
import seaborn as sns
from sklearn.metrics import confusion_matrix
from imblearn.over_sampling import RandomOverSampler
from sklearn.utils import shuffle
from sklearn.neural_network import MLPClassifier

### s02 Schraubvorgänge laden und x,y, usage vorbereiten

In [4]:
data = pyscrew.get_data(scenario="s02", handle_duplicates="first", handle_missings="mean", target_length=800)
df = pd.DataFrame(data)
df.head()

2026-03-22 09:12:00 - INFO - pyscrew.main - Starting data retrieval for scenario: s02 (surface-friction)
2026-03-22 09:12:00 - INFO - pyscrew.pipeline.loading - Using cache directory (absolute): c:\Users\Patrick\miniconda3\envs\thesis_gpu\Lib\site-packages\pyscrew\downloads
2026-03-22 09:12:00 - INFO - pyscrew.pipeline.loading - Beginning data extraction for scenario 's02_variations-in-surface-friction.zip' (force=False)
2026-03-22 09:12:00 - INFO - pyscrew.pipeline.loading - Verifying MD5 checksum for 's02_variations-in-surface-friction.zip'...
2026-03-22 09:12:01 - INFO - pyscrew.pipeline.loading - Checksum verification successful for 's02_variations-in-surface-friction.zip' (MD5: 0bc948a6e8c6e83f72dbe36973131558)
2026-03-22 09:12:01 - INFO - pyscrew.pipeline.loading - Using existing verified file 's02_variations-in-surface-friction.zip' at: c:\Users\Patrick\miniconda3\envs\thesis_gpu\Lib\site-packages\pyscrew\downloads\archives\s02_variations-in-surface-friction.zip
2026-03-22 09:12

2026-03-22 09:13:54 - INFO - pyscrew.pipeline.transformers.handle_missings - Completed missing interpolation using 'mean' method (interval=0.0012)
2026-03-22 09:13:54 - INFO - pyscrew.pipeline.transformers.handle_missings - Processed 12,500 series with 8,619,982 total points
2026-03-22 09:13:54 - INFO - pyscrew.pipeline.transformers.handle_missings - Found gaps - min: 0.0012s, max: 0.1128s, avg: 0.0013s
2026-03-22 09:13:54 - INFO - pyscrew.pipeline.transformers.handle_missings - Added 484,205 points (+5.62% of total)
2026-03-22 09:13:54 - INFO - pyscrew.pipeline.transformers.handle_missings - Average 38.7 points added per series


2026-03-22 09:13:54 - INFO - pyscrew.pipeline.transformers.handle_lengths - Starting to apply equal lengths.
2026-03-22 09:13:54 - INFO - pyscrew.pipeline.transformers.handle_lengths - - 'target_length' : 800
2026-03-22 09:13:54 - INFO - pyscrew.pipeline.transformers.handle_lengths - - 'padding_value' : 0.0
2026-03-22 09:13:54 - INFO - pyscrew.pipeline.transformers.handle_lengths - - 'padding_position' : post
2026-03-22 09:13:54 - INFO - pyscrew.pipeline.transformers.handle_lengths - - 'cutoff_position' : post
2026-03-22 09:13:55 - INFO - pyscrew.pipeline.transformers.handle_lengths - Finished applying equal lengths to the screw driving data.
2026-03-22 09:13:55 - INFO - pyscrew.pipeline.transformers.handle_lengths - - Total screw runs loaded:	12500
2026-03-22 09:13:55 - INFO - pyscrew.pipeline.transformers.handle_lengths - - Average change of length:	728.33 -> 800.00
2026-03-22 09:13:55 - INFO - pyscrew.pipeline.transformers.handle_lengths - - Total points before normalization:	9,104,

,time_values,torque_values,angle_values,gradient_values,step_values,class_values,workpiece_location,workpiece_usage,workpiece_result,scenario_condition,scenario_exception
0,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.067, 0.077, 0.126, 0.087, 0.089, 0.097, 0.0...","[0.5, 1.25, 2.25, 3.75, 5.0, 6.25, 7.5, 8.75, ...","[0.0, 0.0, 0.0298, 0.0214, 0.0064, 0.0023, -0....","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,0,OK,normal,0
1,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.005, 0.008, 0.061, 0.069, 0.104, 0.124, 0....","[0.0, 0.25, 0.75, 1.5, 2.5, 4.0, 5.25, 6.5, 7....","[0.0, 0.0, 0.0, 0.0, 0.0287, 0.0282, 0.0091, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,0,OK,normal,0
2,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, 0.01, 0.039, 0.099, 0.102, 0.087, 0.0...","[0.0, 0.5, 1.25, 2.25, 3.5, 4.75, 6.0, 7.5, 8....","[0.0, 0.0, 0.0, 0.0196, 0.0223, 0.0211, 0.0096...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,1,OK,normal,0
3,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[0.081, 0.059, 0.12, 0.077, 0.043, 0.059, 0.06...","[0.75, 1.75, 2.75, 4.0, 5.25, 6.5, 7.75, 9.25,...","[0.0, 0.0, 0.0261, 0.0207, 0.0, -0.0036, -0.00...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,right,1,OK,normal,0
4,"[0.0, 0.0012, 0.0024, 0.0036, 0.0048, 0.006, 0...","[-0.003, -0.0012, 0.0006, 0.0024, 0.0042, 0.00...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 1.5, 2.5, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.025...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",001_control-group,left,2,OK,normal,0


In [5]:
X = np.array(df['torque_values'].tolist())
y = np.array(df['class_values'].tolist())
usage = np.array(df['workpiece_usage'].tolist())

In [6]:
X.shape, y.shape, usage.shape

((12500, 800), (12500,), (12500,))

In [7]:
print(len(np.unique(y)))
print(np.unique(y, return_counts=True))

8
(array(['001_control-group', '101_used-upper-workpiece',
       '201_lubricant-water', '202_lubricant-oil-based',
       '301_sanding-coarse', '302_sanding-fine', '401_plastic-adhesive',
       '402_surface-chipped'], dtype='<U24'), array([2500, 2500, 1250, 1250, 1250, 1250, 1250, 1250]))


### Folds erstellen und Gridsearch für optim. Hyperparameter

In [8]:
seeds = [1, 42, 56, 73, 81]
split_ratio = 0.65
n_samples = len(X)
split_idx = int(n_samples * split_ratio)

In [9]:
params={'n_estimators': [100, 200, 300, 400, 500]}

for s in seeds:
    X_s, y_s, usage_s = shuffle(X, y, usage, random_state=s)
    X_train, X_test = X_s[:split_idx], X_s[split_idx:]
    y_train, y_test = y_s[:split_idx], y_s[split_idx:]
    usage_train, usage_test = usage_s[:split_idx], usage_s[split_idx:]
    print(f"seed {s} - Trainingsdaten: {len(X_train)} Testdaten: {len(X_test)}")
    
    #Gridsearch f+ür Randomforest
    grid = GridSearchCV(estimator=RandomForestClassifier(random_state=42, class_weight="balanced"), param_grid=params, cv=3, n_jobs=-1, scoring='f1_macro')
    grid.fit(X_train, y_train)
    print(f"Best parameters for seed {s}: {grid.best_params_} mit F1-Score: {grid.best_score_}")


seed 1 - Trainingsdaten: 8125 Testdaten: 4375
Best parameters for seed 1: {'n_estimators': 300} mit F1-Score: 0.3353331736717937
seed 42 - Trainingsdaten: 8125 Testdaten: 4375
Best parameters for seed 42: {'n_estimators': 500} mit F1-Score: 0.3246285165649798
seed 56 - Trainingsdaten: 8125 Testdaten: 4375
Best parameters for seed 56: {'n_estimators': 200} mit F1-Score: 0.3270701781282744
seed 73 - Trainingsdaten: 8125 Testdaten: 4375
Best parameters for seed 73: {'n_estimators': 400} mit F1-Score: 0.33520046639013995
seed 81 - Trainingsdaten: 8125 Testdaten: 4375
Best parameters for seed 81: {'n_estimators': 500} mit F1-Score: 0.32648104938044226


==> Offenbar ein Plateau auf dem ich mich befinde, da die F1-score Werte nur leicht variieren. Ich wähle die Stabile Mitte n_estimators = 400

### Modelltraining

In [10]:
all_folds_results = []

In [11]:
for s in seeds:
    X_s, y_s, usage_s = shuffle(X, y, usage, random_state=s)
    X_train, X_test = X_s[:split_idx], X_s[split_idx:]
    y_train, y_test = y_s[:split_idx], y_s[split_idx:]
    usage_train, usage_test = usage_s[:split_idx], usage_s[split_idx:]
    print(f"seed {s} - Trainingsdaten: {len(X_train)} Testdaten: {len(X_test)}")
    
    #Gridsearch f+ür Randomforest
    rf = RandomForestClassifier(random_state=42, class_weight="balanced", n_estimators=400, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_test)
    
    all_folds_results.append({
        'seed': s+1,
        'True_label':y_test,
        'Pred_label':y_pred_rf,
        'workpiece_usage':usage_test
    })
    print(f"seed {s} - F1-score {f1_score(y_test, y_pred_rf, average='macro')}")

seed 1 - Trainingsdaten: 8125 Testdaten: 4375
seed 1 - F1-score 0.3328094590512283
seed 42 - Trainingsdaten: 8125 Testdaten: 4375
seed 42 - F1-score 0.3613276378841686
seed 56 - Trainingsdaten: 8125 Testdaten: 4375
seed 56 - F1-score 0.34583571943479413
seed 73 - Trainingsdaten: 8125 Testdaten: 4375
seed 73 - F1-score 0.3314910443430656
seed 81 - Trainingsdaten: 8125 Testdaten: 4375
seed 81 - F1-score 0.34663142965232185


In [12]:
all_folds_df = pd.DataFrame(all_folds_results)
all_folds_df.head()

,seed,True_label,Pred_label,workpiece_usage
0,2,"[301_sanding-coarse, 001_control-group, 202_lu...","[401_plastic-adhesive, 001_control-group, 402_...","[22, 21, 10, 16, 8, 2, 15, 14, 13, 4, 20, 4, 1..."
1,43,"[202_lubricant-oil-based, 101_used-upper-workp...","[101_used-upper-workpiece, 101_used-upper-work...","[13, 10, 16, 13, 11, 5, 10, 6, 12, 13, 19, 22,..."
2,57,"[101_used-upper-workpiece, 001_control-group, ...","[001_control-group, 001_control-group, 401_pla...","[6, 7, 4, 8, 24, 8, 9, 21, 18, 7, 14, 0, 20, 2..."
3,74,"[402_surface-chipped, 202_lubricant-oil-based,...","[401_plastic-adhesive, 202_lubricant-oil-based...","[17, 7, 15, 16, 12, 10, 5, 14, 13, 11, 13, 0, ..."
4,82,"[001_control-group, 001_control-group, 101_use...","[402_surface-chipped, 001_control-group, 001_c...","[5, 24, 14, 3, 14, 13, 13, 19, 22, 21, 17, 15,..."


### Auswertung nach Labelstrategie

In [13]:
def pred_filter(fold_results, label="binary", usage1st = True):
    metriken = {"Accuracy": [], "Precision": [], "Recall": [], "F1_macro": []}
    if label =="binary":
        average_method="binary"
    elif label in ["single", "grouped"]:
        average_method="macro"
    
    for fold in fold_results:
        y_val = fold["True_label"]
        y_pred = fold["Pred_label"]
        usage_val = fold["workpiece_usage"]
        if usage1st == True:
            usage_filter = usage_val== 0
            y_val_filtered = y_val[usage_filter]
            y_pred_filtered = y_pred[usage_filter]
        else:
            y_val_filtered = y_val
            y_pred_filtered = y_pred
        if label == "binary":
            y_val_binary = []
            y_pred_binary = []
            for l in y_val_filtered:
                if str(l).startswith('0'):
                    y_val_binary.append(0)
                else:
                    y_val_binary.append(1)
            for l in y_pred_filtered:
                if str(l).startswith('0'):
                    y_pred_binary.append(0)
                else:
                    y_pred_binary.append(1)
            y_val_filtered = np.array(y_val_binary)
            y_pred_filtered = np.array(y_pred_binary)
        elif label == "grouped":
            y_val_group =[]
            y_pred_group=[]
            for l in y_val_filtered:
                if str(l).startswith('0'):
                    y_val_group.append(0)
                elif str(l).startswith('1'):
                    y_val_group.append(1)
                elif str(l).startswith('2') or str(l).startswith('3'):
                    y_val_group.append(2)
                elif str(l).startswith('4') or str(l).startswith('5'):
                    y_val_group.append(3)
                elif str(l).startswith('6') or str(l).startswith('7'):
                    y_val_group.append(4)
            for l in y_pred_filtered:
                if str(l).startswith('0'):
                    y_pred_group.append(0)
                elif str(l).startswith('1'):
                    y_pred_group.append(1)
                elif str(l).startswith('2') or str(l).startswith('3'):
                    y_pred_group.append(2)
                elif str(l).startswith('4') or str(l).startswith('5'):
                    y_pred_group.append(3)
                elif str(l).startswith('6') or str(l).startswith('7'):
                    y_pred_group.append(4)
            y_val_filtered = np.array(y_val_group)
            y_pred_filtered = np.array(y_pred_group)
        elif label == "single":
            y_val_filtered = np.array(y_val_filtered)
            y_pred_filtered = np.array(y_pred_filtered)
        #metriken:acc prec recall f1 mit bib
        metriken["Accuracy"].append(accuracy_score(y_val_filtered, y_pred_filtered))
        metriken["Precision"].append(precision_score(y_val_filtered, y_pred_filtered, average=average_method))
        metriken["Recall"].append(recall_score(y_val_filtered, y_pred_filtered, average=average_method))
        metriken["F1_macro"].append(f1_score(y_val_filtered, y_pred_filtered, average=average_method))

    stats = {"Accuracy_mean": np.mean(metriken["Accuracy"]),"Accuracy_std": np.std(metriken["Accuracy"]), "Precision_mean": np.mean(metriken["Precision"]), "Precision_std": np.std(metriken["Precision"]), "Recall_mean": np.mean(metriken["Recall"]), "Recall_std": np.std(metriken["Recall"]), "F1_macro_mean": np.mean(metriken["F1_macro"]), "F1_macro_std": np.std(metriken["F1_macro"])}
    for i in stats:
        if i != "Accuracy_std" and i != "Precision_std" and i != "Recall_std" and i != "F1_macro_std":
            stats[i] *= 100
    df = pd.DataFrame([stats])
    return df


In [14]:
df_binary_usage1st = pred_filter(all_folds_results, label="binary", usage1st=True)
df_binary_usageall = pred_filter(all_folds_results, label="binary", usage1st=False)
df_single_usage1st = pred_filter(all_folds_results, label="single", usage1st=True)
df_single_usageall = pred_filter(all_folds_results, label="single", usage1st=False)
df_grouped_usage1st = pred_filter(all_folds_results, label="grouped", usage1st=True)
df_grouped_usageall = pred_filter(all_folds_results, label="grouped", usage1st=False)

In [15]:
reproduction_results = pd.concat([df_binary_usage1st, df_binary_usageall, df_single_usage1st, df_single_usageall, df_grouped_usage1st, df_grouped_usageall], ignore_index=True)
#df.insert(loc, column, value)
reproduction_results.insert(0,'Labelstrategie',['Binary_s1st', 'Binary_all', 'Single_s1st', 'Single_all', 'Grouped_s1st', 'Grouped_all'])
reproduction_results

,Labelstrategie,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,F1_macro_mean,F1_macro_std
0,Binary_s1st,81.872717,0.014281,93.986356,0.019446,83.054287,0.026206,88.129250,0.009635
1,Binary_all,73.188571,0.007234,89.220560,0.007003,75.757079,0.010790,81.932637,0.006029
2,Single_s1st,51.206781,0.022637,50.074237,0.032711,49.460808,0.021614,47.801061,0.019723
3,Single_all,37.760000,0.009104,36.632563,0.013398,33.928111,0.009590,34.361906,0.010877
4,Grouped_s1st,65.000810,0.027148,63.928802,0.031170,66.173201,0.025406,63.275212,0.032428
5,Grouped_all,53.892571,0.008192,52.932806,0.009087,52.067380,0.008717,51.233110,0.009105


## Nested Cross Validation

In [16]:
all_seeds_results=[]
inner_seed_val_score=[]

##ußere schicht 0.65 train 0.35 test, 5 mal mit verschiedenen seeds - nach wie vor identisch
for s in seeds:
    X_s, y_s, usage_s = shuffle(X, y, usage, random_state=s)
    X_train_val, X_test = X_s[:split_idx], X_s[split_idx:]
    y_train_val, y_test = y_s[:split_idx], y_s[split_idx:]
    usage_train_val, usage_test = usage_s[:split_idx], usage_s[split_idx:]

    print(f"seed {s} - Trainingsdaten: {len(X_train_val)} Testdaten: {len(X_test)}")

    inner_split = int(len(X_train_val)*0.8)
    X_train, X_val = X_train_val[:inner_split], X_train_val[inner_split:]
    y_train, y_val = y_train_val[:inner_split], y_train_val[inner_split:]
    #innere schicht 0.8 train und 0.2 val
    #Gridsearch f+ür Randomforest
    best_estimator = None
    best_f1_score = 0
    for n in params['n_estimators']:
        rf_inner = RandomForestClassifier(random_state=42, class_weight="balanced", n_estimators=n, n_jobs=-1)
        rf_inner.fit(X_train, y_train)
        y_val_pred = rf_inner.predict(X_val)
        val_f1_score = f1_score(y_val,y_val_pred,average='macro')
        print(f"für seed {s} bei n_estimator {n}: F1-Score: {val_f1_score}")
        if val_f1_score > best_f1_score:
            best_f1_score = val_f1_score
            best_estimator = n

    inner_seed_val_score.append(best_f1_score)
    
    #Finales Training mit den 0.65 train val und 0.35 testdaten mit dem best_estimator als parameter
    rf = RandomForestClassifier(random_state=42, class_weight="balanced", n_estimators=best_estimator, n_jobs=-1) # best_estimator ist bei seed 1 = 100 - f1-score am höchsten bei 0.34357
    rf.fit(X_train_val, y_train_val)
    y_test_pred = rf.predict(X_test)
    
    all_seeds_results.append({
        'seed': s+1,
        'True_label':y_test,
        'Pred_label':y_test_pred,
        'workpiece_usage':usage_test,
        'best_n': best_estimator
    })
    
mean_inner_val = np.mean(inner_seed_val_score)
std_inner_val=np.std(inner_seed_val_score)
print(f"Mean test score für alle seeds: {mean_inner_val} +- {std_inner_val}")


seed 1 - Trainingsdaten: 8125 Testdaten: 4375
für seed 1 bei n_estimator 100: F1-Score: 0.3435750523798121
für seed 1 bei n_estimator 200: F1-Score: 0.337032336588739
für seed 1 bei n_estimator 300: F1-Score: 0.33064598317691335
für seed 1 bei n_estimator 400: F1-Score: 0.3346115724633097
für seed 1 bei n_estimator 500: F1-Score: 0.3386875514971566
seed 42 - Trainingsdaten: 8125 Testdaten: 4375
für seed 42 bei n_estimator 100: F1-Score: 0.3074323757766741
für seed 42 bei n_estimator 200: F1-Score: 0.3035561977326725
für seed 42 bei n_estimator 300: F1-Score: 0.30990214352958206
für seed 42 bei n_estimator 400: F1-Score: 0.31069127953282455
für seed 42 bei n_estimator 500: F1-Score: 0.3068071199154966
seed 56 - Trainingsdaten: 8125 Testdaten: 4375
für seed 56 bei n_estimator 100: F1-Score: 0.31744413296779844
für seed 56 bei n_estimator 200: F1-Score: 0.31701060174946455
für seed 56 bei n_estimator 300: F1-Score: 0.3180481762489528
für seed 56 bei n_estimator 400: F1-Score: 0.3085925919

In [17]:
df_binary_usage1st = pred_filter(all_seeds_results, label="binary", usage1st=True)
df_binary_usageall = pred_filter(all_seeds_results, label="binary", usage1st=False)
df_single_usage1st = pred_filter(all_seeds_results, label="single", usage1st=True)
df_single_usageall = pred_filter(all_seeds_results, label="single", usage1st=False)
df_grouped_usage1st = pred_filter(all_seeds_results, label="grouped", usage1st=True)
df_grouped_usageall = pred_filter(all_seeds_results, label="grouped", usage1st=False)

In [18]:
ncv_results = pd.concat([df_binary_usage1st, df_binary_usageall, df_single_usage1st, df_single_usageall, df_grouped_usage1st, df_grouped_usageall], ignore_index=True)
ncv_results.insert(0,'Labelstrategie',['Binary_s1st', 'Binary_all', 'Single_s1st', 'Single_all', 'Grouped_s1st', 'Grouped_all'])
ncv_results

,Labelstrategie,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,F1_macro_mean,F1_macro_std
0,Binary_s1st,80.897757,0.021020,93.594312,0.020086,82.164360,0.031418,87.445531,0.014555
1,Binary_all,73.033143,0.006956,89.111103,0.007009,75.653616,0.009939,81.827012,0.005825
2,Single_s1st,50.797469,0.023910,49.358889,0.018540,49.121108,0.022764,47.511706,0.019546
3,Single_all,37.664000,0.008865,36.573217,0.012585,33.856013,0.009087,34.302884,0.010173
4,Grouped_s1st,64.226655,0.036281,63.369104,0.043276,65.182950,0.037834,62.547283,0.041679
5,Grouped_all,53.796571,0.008891,52.886279,0.009453,51.946279,0.009435,51.137242,0.009698


## TABPFN Ansatz

In [19]:
import torch
print(torch.cuda.is_available())

True


In [23]:
all_seeds_tabpfn_results=[]
test_scores=[]

for s in seeds:
    X_s, y_s, usage_s = shuffle(X, y, usage, random_state=s)
    X_train_val, X_test = X_s[:split_idx], X_s[split_idx:]
    y_train_val, y_test = y_s[:split_idx], y_s[split_idx:]
    usage_train_val, usage_test = usage_s[:split_idx], usage_s[split_idx:]

    print(f"seed {s} - Trainingsdaten: {len(X_train_val)} Testdaten: {len(X_test)}")
        
    tabpfn = TabPFNClassifier(random_state=42, ignore_pretraining_limits=True, device='cuda', n_estimators=4)
    tabpfn.fit(X_train_val, y_train_val)
    y_test_pred = tabpfn.predict(X_test)
    
    score = f1_score(y_test,y_test_pred,average='macro')
    test_scores.append(score)
    
    all_seeds_tabpfn_results.append({
        'seed': s+1,
        'True_label':y_test,
        'Pred_label':y_test_pred,
        'workpiece_usage':usage_test
    })
print(f"F1-score für alle seeds: {np.mean(test_scores)} +- {np.std(test_scores)}")


seed 1 - Trainingsdaten: 8125 Testdaten: 4375
seed 42 - Trainingsdaten: 8125 Testdaten: 4375
seed 56 - Trainingsdaten: 8125 Testdaten: 4375
seed 73 - Trainingsdaten: 8125 Testdaten: 4375
seed 81 - Trainingsdaten: 8125 Testdaten: 4375
F1-score für alle seeds: 0.3714598573022364 +- 0.008333682445561517


In [24]:
df_binary_usage1st = pred_filter(all_seeds_tabpfn_results, label="binary", usage1st=True)
df_binary_usageall = pred_filter(all_seeds_tabpfn_results, label="binary", usage1st=False)
df_single_usage1st = pred_filter(all_seeds_tabpfn_results, label="single", usage1st=True)
df_single_usageall = pred_filter(all_seeds_tabpfn_results, label="single", usage1st=False)
df_grouped_usage1st = pred_filter(all_seeds_tabpfn_results, label="grouped", usage1st=True)
df_grouped_usageall = pred_filter(all_seeds_tabpfn_results, label="grouped", usage1st=False)

In [25]:
tabpfn_results = pd.concat([df_binary_usage1st, df_binary_usageall, df_single_usage1st, df_single_usageall, df_grouped_usage1st, df_grouped_usageall], ignore_index=True)
tabpfn_results.insert(0,'Labelstrategie',['Binary_s1st', 'Binary_all', 'Single_s1st', 'Single_all', 'Grouped_s1st', 'Grouped_all'])
tabpfn_results

,Labelstrategie,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,F1_macro_mean,F1_macro_std
0,Binary_s1st,84.385583,0.015036,97.011682,0.016554,83.340799,0.011361,89.646867,0.009436
1,Binary_all,73.499429,0.006229,92.379163,0.005157,73.009477,0.006145,81.558413,0.004690
2,Single_s1st,53.160994,0.010604,58.348008,0.028128,51.397095,0.027241,50.209921,0.024073
3,Single_all,40.448000,0.007683,41.256132,0.006594,37.893641,0.008382,37.145986,0.008334
4,Grouped_s1st,69.019846,0.019863,68.152187,0.026458,70.175085,0.023194,66.315317,0.023814
5,Grouped_all,58.985143,0.007489,58.677450,0.010651,56.996991,0.008908,55.032422,0.008350


In [ ]:
#es lief nicht, da ich vergessen hatte huggingface token public zu stellen
#trainingszeit: >100min - Tabpfn ist nicht für so viele daten gemacht. Daher hatte ich bei einer anderen Versionierung zuvor die daten auf insgesamt 5000 verringert (nicht in dieser, hier sind alle daten enthalten)

## MLP Classifier

In [ ]:
all_seeds_mlp_results=[]
mlp_test_score=[]
#MLPClassigier benötigt keine strings als labels sondern integers
le=LabelEncoder()
y_encode =le.fit_transform(y)
for s in seeds:
    X_s, y_s, usage_s = shuffle(X, y_encode, usage, random_state=s)
    X_train, X_test = X_s[:split_idx], X_s[split_idx:] 
    y_train, y_test = y_s[:split_idx], y_s[split_idx:]
    usage_train, usage_test = usage_s[:split_idx], usage_s[split_idx:]
    print(f"seed {s} - Trainingsdaten: {len(X_train)} Testdaten: {len(X_test)}")
    
    #Standardscaler für MLP !!
    scaler=StandardScaler()
    X_train_scale = scaler.fit_transform(X_train)
    X_test_scale = scaler.transform(X_test)
    
    mlp = MLPClassifier(random_state=42, max_iter=500, early_stopping=True,hidden_layer_sizes=(64,32),alpha=0.05) #alpha(regularisierung) zwecks overfitting sonst kommt Precision is ill-defined (da bestimmte klassen nicht gefunden) 
    mlp.fit(X_train_scale, y_train)
    y_pred = mlp.predict(X_test_scale)
    
    score = f1_score(y_test,y_pred,average='macro')
    mlp_test_score.append(score)

    all_seeds_mlp_results.append({
        'seed': s+1,
        'True_label':y_test,
        'Pred_label':y_pred,
        'workpiece_usage':usage_test
    })
print(f"F1-score für alle seeds: {np.mean(mlp_test_score)} +- {np.std(mlp_test_score)}")


seed 1 - Trainingsdaten: 8125 Testdaten: 4375
seed 42 - Trainingsdaten: 8125 Testdaten: 4375
seed 56 - Trainingsdaten: 8125 Testdaten: 4375
seed 73 - Trainingsdaten: 8125 Testdaten: 4375
seed 81 - Trainingsdaten: 8125 Testdaten: 4375
F1-score für alle seeds: 0.32370751455907076 +- 0.009729236702462493


Parameter nur max iteration 500 und early stopping:  
F1-score für alle seeds: 0.32662669939037475 +- 0.012515455889345896  
mit hidden layer sizes 128, 64, 32   
0.33 unt 0.008 std  
mlp = MLPClassifier(random_state=42, max_iter=500, early_stopping=True,hidden_layer_sizes=(256, 128, 64, 32, 16,8))  
F1-score für alle seeds: 0.3340105203929197 +- 0.012038303605863958  
mlp = MLPClassifier(random_state=42, max_iter=500, early_stopping=True,hidden_layer_sizes=(128, 64))  
F1-score für alle seeds: 0.33672820333077724 +- 0.009905468306923125  


In [49]:
df_binary_usage1st = pred_filter(all_seeds_mlp_results, label="binary", usage1st=True)
df_binary_usageall = pred_filter(all_seeds_mlp_results, label="binary", usage1st=False)
df_single_usage1st = pred_filter(all_seeds_mlp_results, label="single", usage1st=True)
df_single_usageall = pred_filter(all_seeds_mlp_results, label="single", usage1st=False)
df_grouped_usage1st = pred_filter(all_seeds_mlp_results, label="grouped", usage1st=True)
df_grouped_usageall = pred_filter(all_seeds_mlp_results, label="grouped", usage1st=False)

In [ ]:
mlp_results = pd.concat([df_binary_usage1st, df_binary_usageall, df_single_usage1st, df_single_usageall, df_grouped_usage1st, df_grouped_usageall], ignore_index=True)
mlp_results.insert(0,'Labelstrategie',['Binary_s1st', 'Binary_all', 'Single_s1st', 'Single_all', 'Grouped_s1st', 'Grouped_all'])
mlp_results

,Labelstrategie,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,F1_macro_mean,F1_macro_std
0,Binary_s1st,80.170974,0.042983,93.374699,0.022448,81.297295,0.056512,86.797061,0.033089
1,Binary_all,75.451429,0.010091,89.170325,0.007265,79.030279,0.019020,83.775957,0.008799
2,Single_s1st,35.382550,0.025103,32.178214,0.065718,30.933580,0.039314,28.513819,0.056373
3,Single_all,35.670857,0.008237,33.490169,0.006859,32.508894,0.010119,32.370751,0.009729
4,Grouped_s1st,46.296085,0.024218,46.619355,0.037856,46.845954,0.035984,43.678475,0.037834
5,Grouped_all,45.321143,0.008749,45.894773,0.006474,45.387617,0.008706,44.908494,0.009667


## Benchmark Auswertung Tabelle alelr Methoden und ihrer Metriken - nach workpieceusage

In [ ]:
reproduction_results['Model']='RandomForest'
reproduction_results
ncv_results['Model']='RandomForest_NCV'
ncv_results
tabpfn_results['Model']='TabPFN'
tabpfn_results
mlp_results['Model']='MLP'
mlp_results

,Labelstrategie,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,F1_macro_mean,F1_macro_std,Model
0,Binary_s1st,80.170974,0.042983,93.374699,0.022448,81.297295,0.056512,86.797061,0.033089,MLP
1,Binary_all,75.451429,0.010091,89.170325,0.007265,79.030279,0.019020,83.775957,0.008799,MLP
2,Single_s1st,35.382550,0.025103,32.178214,0.065718,30.933580,0.039314,28.513819,0.056373,MLP
3,Single_all,35.670857,0.008237,33.490169,0.006859,32.508894,0.010119,32.370751,0.009729,MLP
4,Grouped_s1st,46.296085,0.024218,46.619355,0.037856,46.845954,0.035984,43.678475,0.037834,MLP
5,Grouped_all,45.321143,0.008749,45.894773,0.006474,45.387617,0.008706,44.908494,0.009667,MLP


In [ ]:
benchmark_results = pd.concat([reproduction_results, ncv_results, tabpfn_results, mlp_results], ignore_index=True)
benchmark_results.sort_values(by=['Labelstrategie'],inplace=True)
benchmark_results

,Labelstrategie,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,F1_macro_mean,F1_macro_std,Model
1,Binary_all,73.188571,0.007234,89.220560,0.007003,75.757079,0.010790,81.932637,0.006029,RandomForest
19,Binary_all,75.451429,0.010091,89.170325,0.007265,79.030279,0.019020,83.775957,0.008799,MLP
13,Binary_all,73.499429,0.006229,92.379163,0.005157,73.009477,0.006145,81.558413,0.004690,TabPFN
7,Binary_all,73.033143,0.006956,89.111103,0.007009,75.653616,0.009939,81.827012,0.005825,RandomForest_NCV
0,Binary_s1st,81.872717,0.014281,93.986356,0.019446,83.054287,0.026206,88.129250,0.009635,RandomForest
18,Binary_s1st,80.170974,0.042983,93.374699,0.022448,81.297295,0.056512,86.797061,0.033089,MLP
6,Binary_s1st,80.897757,0.021020,93.594312,0.020086,82.164360,0.031418,87.445531,0.014555,RandomForest_NCV
12,Binary_s1st,84.385583,0.015036,97.011682,0.016554,83.340799,0.011361,89.646867,0.009436,TabPFN
17,Grouped_all,58.985143,0.007489,58.677450,0.010651,56.996991,0.008908,55.032422,0.008350,TabPFN
11,Grouped_all,53.796571,0.008891,52.886279,0.009453,51.946279,0.009435,51.137242,0.009698,RandomForest_NCV


## PIML
drehmoment formel screw driving M = k * d * f  
M = Torque (vorhanden)  
k = Reibungskoeffizient   
d = Durchmesser Schraube (vorhanden)  DELTA PT® 40x12 4.0mm durchmesser x 12mm länge  
F = Vorspannkraft (nicht wirklich vorhanden - oder?)  
